# DSC670-T301 Advanced Uses of Generative AI 

### Submitted by - Rethish Plappuzha Sreedharan Nair


## Week 2 Exercise
 <p>
Using the OpenAI API, extract the text from this invoice (dsc-670-exercise-invoice.pdf) and then use an LLM to convert the text to structured JSON (you can literally ask the LLM to produce JSON).
 </p>

In [55]:
# Importing the necessary libraries
import os
from openai import OpenAI
from dotenv import load_dotenv

# Loading environment variables from the .env file for the OpenAI API key
load_dotenv()


if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Please set the OPENAI_API_KEY environment variable before running this cell.")

# Setting up the OpenAI API key
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [56]:
# Reading the content of the file 'dsc-670-exercise-invoice.pdf' and converting it to text
from PyPDF2 import PdfReader

def extract_text_from_pdf(path: str) -> str:
    with open(path, "rb") as f:
        reader = PdfReader(f)
        pages = [p.extract_text() or "" for p in reader.pages]
    return "\n".join(pages)

In [57]:

from pyparsing import Any, Dict
import json
import re

# Function to call the LLM and extract invoice information as a JSON object
# args:
# - text: The raw text extracted from the invoice PDF.
# - model: The name of the OpenAI model to use (default is "gpt-4o").

def call_llm_for_invoice_json(text: str, model: str = "gpt-4o"):
    system_msg = {
        "role": "system",
        "content": "You are an assistant that extracts invoice information and returns a single valid JSON object only."
    }
    user_msg = {
        "role": "user",
        "content": (
            f"Extract structured invoice data from the following text and return a single JSON object. {text}"
            
        )
    }
    # Call the OpenAI API to get the model's response.
    # With temperature set to 0 for deterministic output, meaning the model will produce the same output for the same input.

    resp = client.chat.completions.create(
        model=model,
        messages=[system_msg, user_msg],
        temperature=0
    )

    # Print the token usage and number of tokens for debugging and cost estimation
    print(f"Token usage: {resp.usage}")
    print(f"Number of tokens: {resp.usage.total_tokens}")

    raw = resp.choices[0].message.content
    print("Raw model output:")
    print(raw)
    print("-" * 40)

    # Strip code fences if present and extract first JSON object
    m = re.search(r"```(?:json)?\s*(\{.*\})\s*```", raw, re.S)
    candidate = m.group(1) if m else raw
    if not m:
        m2 = re.search(r"(\{.*\})", candidate, re.S)
        if m2:
            candidate = m2.group(1)

    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        # If parsing failed, raise with model output for debugging
        raise ValueError(f"Failed to parse JSON from model output:\n{raw}")


In [58]:
pdf_path = "./dsc-670-exercise-invoice.pdf" # Path to the invoice PDF file.
# Reading the PDF file and extracting text from it using the defined function.
text = extract_text_from_pdf(pdf_path)
print(text)

321 Avenue A Date: 6/28/2024
Portland, OR 12345 Invoice # 1111
Phone: (206) 555-1163 For PO # 123456
Fax: (206) 555-1164
someone@websitegoeshere.com
Quantity Description Unit price Amount Discount applied
1 Item Number 1 2.00 $                                       2.00 $                                       
1 Item Number 2 2.00 $                                       2.00 $                                       
1 Item Number 3 2.00 $                                       2.00 $                                       
- $                                         
- $                                         
- $                                         
- $                                         
- $                                         
- $                                         
- $                                         
- $                                         
Subtotal 6.00 $                                       
Credit 1,000.00 $                                
Tax 9.80%

In [59]:
invoice_json = call_llm_for_invoice_json(text, model="gpt-4o")

Token usage: CompletionUsage(completion_tokens=377, prompt_tokens=322, total_tokens=699, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))
Number of tokens: 699
Raw model output:
```json
{
  "invoice_number": "1111",
  "date": "6/28/2024",
  "billing_address": {
    "name": "Natasha Jones",
    "company": "Central Beauty",
    "address": "123 Main St.",
    "city": "Manhattan",
    "state": "NY",
    "zip": "98765",
    "phone": "(321) 555-1234"
  },
  "company_info": {
    "address": "321 Avenue A",
    "city": "Portland",
    "state": "OR",
    "zip": "12345",
    "phone": "(206) 555-1163",
    "fax": "(206) 555-1164",
    "email": "someone@websitegoeshere.com"
  },
  "purchase_order_number": "123456",
  "items": [
    {
      "quantity": 1,
      "description": "Item Number 1",
      "unit_price": 2.00,
      "am

In [60]:
# Display result
print(json.dumps(invoice_json, indent=2))

{
  "invoice_number": "1111",
  "date": "6/28/2024",
  "billing_address": {
    "name": "Natasha Jones",
    "company": "Central Beauty",
    "address": "123 Main St.",
    "city": "Manhattan",
    "state": "NY",
    "zip": "98765",
    "phone": "(321) 555-1234"
  },
  "company_info": {
    "address": "321 Avenue A",
    "city": "Portland",
    "state": "OR",
    "zip": "12345",
    "phone": "(206) 555-1163",
    "fax": "(206) 555-1164",
    "email": "someone@websitegoeshere.com"
  },
  "purchase_order_number": "123456",
  "items": [
    {
      "quantity": 1,
      "description": "Item Number 1",
      "unit_price": 2.0,
      "amount": 2.0
    },
    {
      "quantity": 1,
      "description": "Item Number 2",
      "unit_price": 2.0,
      "amount": 2.0
    },
    {
      "quantity": 1,
      "description": "Item Number 3",
      "unit_price": 2.0,
      "amount": 2.0
    }
  ],
  "subtotal": 6.0,
  "credit": 1000.0,
  "tax_rate": 9.8,
  "additional_discount": 12,
  "balance_due": -

### Analysis
<p>
The OpenAI API is used to extract invoice information from the PDF file and generate a JSON output.
</p>
<p>
The OpenAI API client is being created using the KEY, and we use it to communicate with the LLM. The function "call_llm_for_invoice_json" is used for calling the LLM and generating the output.
</p>
<p>
The function we are passing the model and the text from the PDF file in raw format. The model we use in this case is "gpt-4o," meaning the LLM will process using this model. This GPT-4o model is more capable than the GPT-4 and GPT-4 Turbo, but it is also twice as fast and 50% cheaper." I would like to try different models and see if the outputs differ.
</p>
<p>
Let's go deeper into the LLM calling function, where we create the Completions object by passing the model information and the messages list. The experiment we are doing is passing very little information about the text we are passing and how it will interpret it.

We pass two messages (A list of message parameters, where each message is used to construct the dialogue or prompt for the model) as user and system roles, which will help the LLM understand the input and generate the output.

We are passing the temperature argument as 0 to generate a more deterministic output. A low temperature, such as zero, helps select the highest-probability tokens, making it ideal for code generation and data extraction.

```
 resp = client.chat.completions.create(
        model=model,
        messages=[system_msg, user_msg],
        temperature=0
    )
```
</p>
<p>
The function will use the completion object to generate responses to the input messages. We can see that the output clearly reads the input text and transforms it to JSON. The function also prints the number of tokens used for generating the output. I will be doing a deeper look into the completion object and additional information we can get regarding the token processing and how the model interprets, etc., in the latter part of the course.
</p>
